# Bài học 11 - Giao thức Đại lý-đến-Đại lý (A2A)


## Thiết lập


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Giao thức A2A là gì?

**Giao thức Agent-to-Agent (A2A)** là một tiêu chuẩn mở cho phép các đại lý AI giao tiếp,
phát hiện lẫn nhau và hợp tác — ngay cả khi chúng được xây dựng trên các khung khác nhau hoặc được lưu trữ
bởi các dịch vụ khác nhau.

Các khái niệm chính:

- **Khám phá** – Các đại lý công bố một *Thẻ Đại lý* mô tả khả năng của họ, giúp
  các đại lý khác (hoặc người điều phối) dễ dàng tìm ra chuyên gia phù hợp cho một nhiệm vụ.
- **Truyền tin nhắn** – Các đại lý trao đổi các tin nhắn có cấu trúc thông qua một giao thức chung, để một
  yêu cầu từ một đại lý có thể được hiểu và thực hiện bởi một đại lý khác bất kể
  cách triển khai nội bộ của nó.
- **Vòng đời nhiệm vụ** – A2A định nghĩa các trạng thái như *đã gửi*, *đang làm*, *hoàn thành*, và
  *thất bại*, cho phép người điều phối theo dõi đầy đủ tiến trình của một nhiệm vụ được giao.

Trong bài học này, chúng ta mô phỏng sự hợp tác kiểu A2A bằng cách kết nối ba đại lý du lịch chuyên biệt
vào một quy trình làm việc, trong đó mỗi đại lý đóng góp chuyên môn của mình và chuyển kết quả cho đại lý tiếp theo.


## Tạo các Đại lý Du lịch Chuyên biệt


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Hợp tác đa tác tử qua quy trình làm việc

Chúng tôi kết nối ba tác tử vào một quy trình làm việc tuần tự phản ánh việc truyền thông điệp A2A:

1. **CurrencyExchangeAgent** nhận yêu cầu của người dùng và tạo hướng dẫn về tiền tệ.
2. **ActivityPlannerAgent** nhận ngữ cảnh được làm giàu và thêm các đề xuất hoạt động.
3. **TravelManagerAgent** tổng hợp cả hai đầu vào thành bản tóm tắt chuyến đi cuối cùng.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Hiểu về A2A trong Môi trường Sản xuất

Trong môi trường sản xuất, giao thức A2A mở khóa các kịch bản mạnh mẽ giữa các dịch vụ:

| Khả năng | Mô tả |
|---|---|
| **Tương tác giữa các framework** | Một agent được xây dựng với một framework có thể uỷ quyền nhiệm vụ cho agent được xây dựng với bất kỳ framework nào khác tuân thủ A2A, cho phép tương tác thực sự giữa các tổ chức. |
| **Ranh giới dịch vụ** | Các agent có thể nằm trong các microservice riêng biệt, vùng đám mây, hoặc thậm chí các tổ chức khác nhau trong khi vẫn cộng tác một cách liền mạch. |
| **Khám phá động** | Một bộ điều phối có thể truy vấn đăng ký Agent Card khi chạy để tìm chuyên gia phù hợp nhất cho một nhiệm vụ phụ cụ thể. |
| **Phát trực tiếp & thông báo đẩy** | A2A hỗ trợ Server-Sent Events (SSE) để cập nhật tiến trình theo thời gian thực và thông báo đẩy cho các tác vụ chạy lâu. |

Quy trình làm việc chúng ta xây dựng ở trên là một phiên bản đơn giản, chạy trong cùng tiến trình của mẫu này. Trong triển khai thực tế,
mỗi agent sẽ cung cấp một endpoint HTTP, xuất bản một Agent Card, và giao tiếp
qua giao thức A2A JSON-RPC.


## Tóm tắt

Trong bài học này bạn đã học:

1. **Giao thức A2A là gì** — một tiêu chuẩn mở cho việc khám phá, nhắn tin,
   và quản lý nhiệm vụ giữa các đại lý.
2. **Cách tạo đại lý chuyên biệt** — một đại lý trao đổi tiền tệ, một đại lý lập kế hoạch hoạt động,
   và một trình điều phối Quản lý Du lịch.
3. **Cách kết nối các đại lý vào quy trình làm việc** — sử dụng `WorkflowBuilder` để mô hình hóa việc truyền thông điệp tuần tự giữa các đại lý.

4. **Cách A2A hoạt động trong môi trường sản xuất** — cho phép hợp tác giữa các framework, dịch vụ khác nhau
   với việc khám phá động và cập nhật luồng.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
